# 01. 데이터 전처리

제10차 산업안전보건 실태조사(2021) 원자료 → 최종 분석 데이터셋(n=1,375, 17변수) 생성 과정.

**KEY PAPER 대응**: Exclusion Criteria + Data Pre-processing

## 종속변수 구성 (Q27_3_1/2/3 → 사고발생)

원자료의 사고 관련 3개 문항(부상 4~89일·90일 이상·사망)을 통합하여 이진 종속변수 `사고발생`을 생성한다. 본 셀은 재현성 확보용으로, 결과는 다음 셀에서 기존 `전처리_최종.csv`의 `사고발생` 컬럼과 100% 일치함을 검증한다.

In [1]:
# 원자료에서 종속변수 구성 로직 재현
import pandas as pd

raw = pd.read_csv('../data/제10차 산업안전보건 실태조사_raw data_건설업_230824.CSV', encoding='cp949')
print(f'원자료: {raw.shape}')

# 제외 기준 적용 (기존 1,375개 표본과 일치시키기)
df_temp = raw.copy()
target_cols = ['Q27_3_1', 'Q27_3_2', 'Q27_3_3']

# 단계 1: 종속변수 전체 결측 제거
df_temp = df_temp[~df_temp[target_cols].isna().all(axis=1)]
# 단계 2: 이상치 (사망 30건) 제거
df_temp = df_temp[df_temp['Q27_3_3'] != 30]
# 단계 3~6: 무응답 사례 제거
df_temp = df_temp[df_temp['Q6'] != 9]
df_temp = df_temp[~df_temp['Q10'].isin([4, 9])]
df_temp = df_temp[df_temp['Q10_1'] != 9]
df_temp = df_temp[~df_temp['Q14'].isna()]
df_temp = df_temp[df_temp['Q9'] != 9]

print(f'제외 후: {len(df_temp)}개 (목표: 1375개)')
assert len(df_temp) == 1375, f'표본 크기 불일치: {len(df_temp)}'

# 종속변수 이진화
df_temp['사고발생'] = (
    (df_temp['Q27_3_1'].fillna(0) > 0) |
    (df_temp['Q27_3_2'].fillna(0) > 0) |
    (df_temp['Q27_3_3'].fillna(0) > 0)
).astype(int)

print(f"사고발생 분포: {df_temp['사고발생'].value_counts().to_dict()}")
print(f"사고발생 비율: {df_temp['사고발생'].mean():.3f}")

원자료: (1502, 181)
제외 후: 1375개 (목표: 1375개)
사고발생 분포: {0: 984, 1: 391}
사고발생 비율: 0.284


In [2]:
# 기존 전처리_최종.csv의 사고발생과 비교 검증
existing = pd.read_csv('../data/전처리_최종.csv')

# 인덱스 재정렬 후 비교
df_temp_sorted = df_temp.reset_index(drop=True)
existing_sorted = existing.reset_index(drop=True)

matches = (df_temp_sorted['사고발생'].values == existing_sorted['사고발생'].values).sum()
total = len(existing_sorted)

print(f'사고발생 일치: {matches}/{total} ({matches/total*100:.1f}%)')
assert matches == total, f'사고발생 불일치: {total - matches}건'
print('✅ 종속변수 재현 로직이 기존 CSV와 완전 일치합니다.')

사고발생 일치: 1375/1375 (100.0%)
✅ 종속변수 재현 로직이 기존 CSV와 완전 일치합니다.


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 원자료 로드 (이미 전처리 완료된 파일 사용)
df = pd.read_csv('../data/전처리_최종.csv')
print(f'최종 데이터셋: {df.shape[0]}개 사업장, {df.shape[1]}개 변수')

최종 데이터셋: 1375개 사업장, 17개 변수


## 1. 제외 과정 요약

| 단계 | 제거 대상 | 제거 수 | 잔여 |
|:---:|---|:---:|:---:|
| 시작 | 원자료 | — | 1,502 |
| 1 | 종속변수 판단 불가 | 16 | 1,486 |
| 2 | 종속변수 이상치 (사망 30건) | 1 | 1,485 |
| 3 | 안전조직 무응답 | 21 | 1,464 |
| 4 | 위원회 무응답/불명 | 62 | 1,402 |
| 5 | 위험성평가 구조적 미응답 | 24 | 1,378 |
| 6 | 전문지도 무응답 | 3 | **1,375** |

**결측 처리**: KNN imputation(KEY PAPER) 대신 Listwise Deletion 채택.
- 이유: 본 데이터의 결측은 설문 무응답/조건부 미응답(Skip Logic)으로, 무응답 자체가 정보를 담고 있어 인위적 대체는 편향 유발.

## 2. 종속변수 분포

In [4]:
target_dist = df['사고발생'].value_counts().sort_index()
for val, cnt in target_dist.items():
    label = '사고 미발생' if val == 0 else '사고 발생'
    print(f'  {val} ({label}): {cnt}개 ({cnt/len(df)*100:.1f}%)')

  0 (사고 미발생): 984개 (71.6%)
  1 (사고 발생): 391개 (28.4%)


## 3. 변수 구성 확인

In [5]:
GROUP_A = ['안전조직수준', '위원회수준', '인증보유']
GROUP_B = ['위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여']
MODERATORS = ['전문지도', '고용노동부감독', '안전보건공단지원']
CONTROLS = ['공사규모', '발주처', '기성공정률', '공사종류', '외국인비율']
TARGET = ['사고발생']

print(f'독립변수 A (내부관리): {GROUP_A}')
print(f'독립변수 B (현장행동): {GROUP_B}')
print(f'조절변수: {MODERATORS}')
print(f'통제변수: {CONTROLS}')
print(f'종속변수: {TARGET}')
print(f'\n총 {len(GROUP_A)+len(GROUP_B)+len(MODERATORS)+len(CONTROLS)+1}개 변수')

독립변수 A (내부관리): ['안전조직수준', '위원회수준', '인증보유']
독립변수 B (현장행동): ['위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여']
조절변수: ['전문지도', '고용노동부감독', '안전보건공단지원']
통제변수: ['공사규모', '발주처', '기성공정률', '공사종류', '외국인비율']
종속변수: ['사고발생']

총 17개 변수


## 4. 기술통계

In [6]:
CATEGORICAL = ['인증보유', '공사규모', '발주처', '공사종류']
CONTINUOUS = [c for c in df.columns if c not in CATEGORICAL + ['사고발생']]

table1 = pd.DataFrame({
    '평균±표준편차': [f"{df[c].mean():.2f}±{df[c].std():.2f}" for c in CONTINUOUS],
    '최소값': [df[c].min() for c in CONTINUOUS],
    'Q1': [df[c].quantile(0.25) for c in CONTINUOUS],
    '중앙값': [df[c].median() for c in CONTINUOUS],
    'Q3': [df[c].quantile(0.75) for c in CONTINUOUS],
    '최대값': [df[c].max() for c in CONTINUOUS],
}, index=CONTINUOUS)
table1.index.name = '변수명'
table1

,평균±표준편차,최소값,Q1,중앙값,Q3,최대값
변수명,,,,,,
안전조직수준,0.98±0.15,0.0,1.0,1.0,1.00,1.0
위원회수준,0.73±0.44,0.0,0.0,1.0,1.00,1.0
위험성평가수준,1.78±0.59,0.0,2.0,2.0,2.00,2.0
교육훈련도움,4.31±0.74,1.0,4.0,4.0,5.00,5.0
정리정돈상태,4.22±0.76,1.0,4.0,4.0,5.00,5.0
작업중지권,4.35±0.75,1.0,4.0,4.0,5.00,5.0
작업반장기여,4.13±0.82,1.0,4.0,4.0,5.00,5.0
전문지도,0.36±0.48,0.0,0.0,0.0,1.00,1.0
고용노동부감독,0.51±0.50,0.0,0.0,1.0,1.00,1.0


## 5. 범주형 분포

In [7]:
for c in CATEGORICAL + ['사고발생']:
    print(f'\n[{c}]')
    counts = df[c].value_counts().sort_index()
    for cat, n in counts.items():
        print(f'  {cat}: {n}개 ({n/len(df)*100:.1f}%)')


[인증보유]
  0: 937개 (68.1%)
  1: 438개 (31.9%)

[공사규모]
  1: 414개 (30.1%)
  2: 634개 (46.1%)
  3: 327개 (23.8%)

[발주처]
  1: 501개 (36.4%)
  2: 719개 (52.3%)
  3: 155개 (11.3%)

[공사종류]
  1: 424개 (30.8%)
  2: 59개 (4.3%)
  3: 287개 (20.9%)
  4: 241개 (17.5%)
  5: 124개 (9.0%)
  6: 149개 (10.8%)
  7: 91개 (6.6%)

[사고발생]
  0: 984개 (71.6%)
  1: 391개 (28.4%)


## 6. 전처리 완료

최종 데이터셋 `data/전처리_최종.csv` (1,375 × 17)이 분석에 사용된다.